## Import Necessary Libraries

In [1]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

## Scaled Dot Product Attention

In [ ]:
def scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    mask: torch.Tensor | None = None,
    dropout: nn.Dropout | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute Scaled Dot-Product Attention.

    Args:
        query:   (batch, heads, seq_len_q, d_k)
        key:     (batch, heads, seq_len_k, d_k)
        value:   (batch, heads, seq_len_k, d_v)
        mask:    Broadcastable mask. 0/False positions are *allowed*,
                 True positions are *blocked* (filled with -inf before softmax).
        dropout: Optional dropout applied to the attention weights.

    Returns:
        output:  (batch, heads, seq_len_q, d_v)
        attn_weights: (batch, heads, seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)

    # Step 1: Compute raw attention scores  →  Q K^T
    # Shape: (batch, heads, seq_len_q, seq_len_k)
    scores = torch.matmul(query, key.transpose(-2, -1))

    # Step 2: Scale by √d_k to stabilise gradients
    scores = scores / math.sqrt(d_k)

    # Step 3: Apply mask (used for padding and causal/autoregressive decoding)
    # Masked positions get -inf so that softmax assigns them ~0 probability.
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 4: Softmax along the key dimension → attention weights sum to 1
    attn_weights = F.softmax(scores, dim=-1)

    # Step 5: Optional dropout on attention weights (Section 5.4 in the paper)
    if dropout is not None:
        attn_weights = dropout(attn_weights)

    # Step 6: Weighted sum of values
    output = torch.matmul(attn_weights, value)

    return output, attn_weights